Ground Truth for test images were not provided. That is why we use the train dataset as the dataset on why we do a train-test-split. 

In [1]:
import shutil
from pathlib import Path

from tifffile import TiffFile
import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit


In [ ]:
ROOT=Path("../")

(ROOT / "local_dataset").mkdir(parents=True, exist_ok=True)
(ROOT / "local_dataset/csv_files").mkdir(parents=True, exist_ok=True)

In [3]:
df = pd.read_csv("../train_labels_GY1QjFw.csv")

# Colonnes contenant les classes
class_cols = [
    c for c in df.columns
    if c not in ["sample_id", "no_data", "clouds"]
]

# Matrice binaire :
# 1 = classe présente dans l'image
# 0 = classe absente de l'image

Y = (df[class_cols].fillna(0).to_numpy() > 0).astype(int)


X = df["sample_id"]

# Première séparation : 80% train / 20% temporaire
msss_1 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2, 
    random_state=42
)


train_idx, temp_idx = next(msss_1.split(X, Y))


X_temp = X.iloc[temp_idx]
Y_temp = Y[temp_idx]

msss_2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_rel_idx, test_rel_idx = next(
    msss_2.split(X_temp, Y_temp)
)

train_ids = X.iloc[train_idx]
val_ids = X_temp.iloc[val_rel_idx]
test_ids = X_temp.iloc[test_rel_idx]

In [4]:
CSV_DIR= Path(ROOT / "local_dataset/csv_files")

train_ids.to_csv(CSV_DIR/"train_images.csv", index=False)
val_ids.to_csv(CSV_DIR/"validation_images.csv", index=False)
test_ids.to_csv(CSV_DIR/"test_images.csv", index=False)

print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

Train: 14794
Validation: 1848
Test: 1849


In [5]:
train_df = df.iloc[train_idx].copy()
val_df = df.iloc[temp_idx[val_rel_idx]].copy()
test_df = df.iloc[temp_idx[test_rel_idx]].copy()


train_df.to_csv(CSV_DIR/"train_labels.csv", index=False)
val_df.to_csv(CSV_DIR/"validation_labels.csv", index=False)
test_df.to_csv(CSV_DIR/"test_labels.csv", index=False)

In [6]:
LOCAL_DATASET_DIR=Path(ROOT / "local_dataset")
DATASET_FOLDER = Path("../dataset_UNZIPPED")


# Train
train_images_paths = [DATASET_FOLDER / f"train/images/{sid}.tif" for sid in train_df["sample_id"]]
train_masks_paths = [DATASET_FOLDER / f"train/masks/{sid}.tif" for sid in train_df["sample_id"]]

# Validation
val_images_paths = [DATASET_FOLDER / f"train/images/{sid}.tif" for sid in val_df["sample_id"]]
val_masks_paths = [DATASET_FOLDER / f"train/masks/{sid}.tif" for sid in val_df["sample_id"]]

# Test
test_images_paths = [DATASET_FOLDER / f"train/images/{sid}.tif" for sid in test_df["sample_id"]]



In [7]:
for split, images, masks in [
    ("train", train_images_paths, train_masks_paths),
    ("validation", val_images_paths, val_masks_paths),
    ("test", test_images_paths, None)
]:
    (LOCAL_DATASET_DIR / split / "images").mkdir(parents=True, exist_ok=True)

    for img in images:
        shutil.copy(img, LOCAL_DATASET_DIR / split / "images" / img.name)

    if masks is not None:
        (LOCAL_DATASET_DIR / split / "masks").mkdir(parents=True, exist_ok=True)

        for mask in masks:
            shutil.copy(mask, LOCAL_DATASET_DIR / split / "masks" / mask.name)

In [8]:
assert set(train_ids).isdisjoint(set(val_ids))
assert set(train_ids).isdisjoint(set(test_ids))
assert set(val_ids).isdisjoint(set(test_ids))

print("Aucun chevauchement entre les trois ensembles.")

Aucun chevauchement entre les trois ensembles.


In [9]:
# Number of images per set

print(f"Train: {len(train_images_paths)}")
print(f"Val: {len(val_images_paths)}")
print(f"Test: {len(test_images_paths)}")

Train: 14794
Val: 1848
Test: 1849


In [10]:
# Max & min value pixel in dataset

global_min = float("inf")
global_max = float("-inf")

for path in train_images_paths:# + val_images_paths:
    with TiffFile(path) as t:
        img = t.asarray()

    global_min = min(global_min, img.min())
    global_max = max(global_max, img.max())

print("Minimum global :", global_min)
print("Maximum global :", global_max)

Minimum global : 1
Maximum global : 24356


In [11]:
TRAIN_CLASS_COUNTS = np.zeros(10, dtype=np.int64)
VAL_CLASS_COUNTS = np.zeros(10, dtype=np.int64)

for path in train_masks_paths:

    with TiffFile(path) as t:
        mask = t.asarray()

    for class_id in range(10):
        TRAIN_CLASS_COUNTS[class_id] += np.sum(mask == class_id)
        
        
for path in val_masks_paths:

    with TiffFile(path) as t:
        mask = t.asarray()

    for class_id in range(10):
        VAL_CLASS_COUNTS[class_id] += np.sum(mask == class_id)

# statistics
# the pixel class counts in the training set
print(f"TRAIN_CLASS_COUNTS: {TRAIN_CLASS_COUNTS}")
print(f"VAL_CLASS_COUNTS: {VAL_CLASS_COUNTS}")

TRAIN_CLASS_COUNTS: [        0     17441  48775625 323881666 222414642  76948228 265937117
   7749139       828  23814898]
VAL_CLASS_COUNTS: [       0     1249  6258733 40140226 27516024 10176292 33510997  1030736
       23  2476248]
